# Автоматическая семантическая разметка CoNLL-U с RuBERT

Пайплайн устойчив к разрывам соединения: показывает прогресс длительных этапов и сохраняет возобновляемые checkpoints на Google Drive. При повторном запуске готовые пакеты загружаются с диска.

Источники данных и набор тегов взяты из `03_WSD_(НКРЯ,_осн_корпус)+multi_classification_ipynb_.ipynb`; исходный ноутбук не изменяется.


In [1]:
# Для чистого Google Colab выполните один раз.
%pip install -q transformers sentencepiece conllu datasets tqdm pandas scikit-learn


## 1. Подключение Google Drive и каталоги кэша

Все checkpoints хранятся на Drive. Не удаляйте `rubert_semantic_checkpoints`, если хотите продолжить оборванный запуск.


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset, load_from_disk
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Запуск вне Google Colab: задайте LOCAL_ROOT ниже.')

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Аспирантура')
CHECKPOINT_ROOT = DRIVE_ROOT / 'rubert_semantic_checkpoints'
MODEL_CACHE = CHECKPOINT_ROOT / 'huggingface_models'
DATASET_CACHE = CHECKPOINT_ROOT / 'dictionary_dataset'
VECTOR_CACHE = CHECKPOINT_ROOT / 'vectors'
CORPUS_CACHE = CHECKPOINT_ROOT / 'corpus'
for directory in (MODEL_CACHE, VECTOR_CACHE, CORPUS_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

CONLLU_FILE = DRIVE_ROOT / 'conllu/handlabeled/GramEval2020-test-fiction.conllu'
OUTPUT_FILE = DRIVE_ROOT / 'conllu/handlabeled/GramEval2020-test-fiction-rubert-sem.conllu'
DIAGNOSTICS_FILE = DRIVE_ROOT / 'conllu/handlabeled/GramEval2020-test-fiction-rubert-sem.pkl'
print('Checkpoints:', CHECKPOINT_ROOT)


Mounted at /content/drive
Checkpoints: /content/drive/MyDrive/Colab Notebooks/Аспирантура/rubert_semantic_checkpoints


## 2. Загрузка CoNLL-U и словаря с локальным кэшем

In [3]:
with CONLLU_FILE.open('r', encoding='utf-8') as file:
    conllu_data = file.read()
print(f'CoNLL-U загружен: {len(conllu_data):,} символов')

if (DATASET_CACHE / 'dataset_dict.json').exists():
    print('Загрузка словаря из Google Drive...')
    dictionary_definitions = load_from_disk(str(DATASET_CACHE))
else:
    print('Первая загрузка словаря с Hugging Face...')
    dictionary_definitions = load_dataset(
        'snagbreac/russian-reverse-dictionary-train-data',
        download_mode='reuse_cache_if_exists',
    )
    dictionary_definitions.save_to_disk(str(DATASET_CACHE))
    print('Словарь сохранён на Google Drive')


CoNLL-U загружен: 99,435 символов
Первая загрузка словаря с Hugging Face...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/294007 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/294007 [00:00<?, ? examples/s]

Словарь сохранён на Google Drive


## 3. Набор семантических тегов

In [4]:
TAGS = {
    "abstr": "абстрактные понятия, не имеющие физической формы",
    "age": "возраст и возрастные характеристики",
    "animal": "животные и живые существа животного мира",
    "appear": "появление, возникновение чего-либо",
    "artwork": "произведения искусства как объекты творчества",
    "art": "искусство как деятельность и культурная сфера",
    "be": "существование, бытие, факт наличия",
    "behav": "поведение и модели действий",
    "berry": "ягоды как биологический класс плодов",
    "changest": "изменение состояния или характеристик",
    "cin": "кинематограф, кино как индустрия и искусство",
    "class": "класс, категория или группа объектов",
    "cloth": "одежда и текстильные изделия",
    "color": "цветовые характеристики и восприятие цвета",
    "concr": "конкретные предметные объекты",
    "constr": "здания и строительные сооружения",
    "contact": "контактирование и взаимодействие через соприкосновение или связь",
    "degr": "степень, мера интенсивности или выраженности",
    "device": "высокотехнологичные устройства и автоматические системы",
    "dir": "направление движения или ориентации",
    "disappear": "исчезновение или прекращение существования",
    "disease": "болезни и патологические состояния",
    "dish": "посуда и предметы кухонного использования",
    "dist": "расстояние между объектами",
    "doc": "документы и текстовые официальные материалы",
    "dram": "театр и драматическое искусство",
    "emot": "эмоции и эмоциональные состояния",
    "etn": "этнонимы и названия народов",
    "ev": "оценка, результат оценивания",
    "famn": "фамилии как имена семейного происхождения",
    "food": "еда и пищевые продукты",
    "form": "форма и структурные характеристики",
    "fruit": "плоды растений",
    "furn": "мебель и предметы обстановки",
    "grain": "зерновые культуры и зерно",
    "group": "группы людей или объединения",
    "hum": "человек как биологический и социальный субъект",
    "humq": "качества человека и личностные характеристики",
    "impact": "воздействие одного объекта на другой",
    "instr": "инструменты как средства выполнения действий",
    "inter": "взаимодействие между объектами или субъектами",
    "kin": "родственные отношения",
    "light": "свет и световые явления",
    "max": "большая степень, максимум значения",
    "ment": "ментальная и когнитивная сфера",
    "min": "малое значение, минимум",
    "move": "передвижение и движение",
    "mus": "музыка как искусство и звуковая система",
    "neg": "негативная оценка или отрицательная характеристика",
    "org": "организации и институциональные структуры",
    "part": "часть целого",
    "patrn": "отчество как часть имени",
    "perc": "восприятие и перцептивные процессы",
    "physiol": "физиологические процессы живых организмов",
    "physq": "физические свойства объектов",
    "place": "место и пространственные объекты",
    "plant": "растения и ботанические объекты",
    "posit": "позитивная оценка или положительная характеристика",
    "poss": "посессивная сфера, принадлежность",
    "prof": "профессии и виды профессиональной деятельности",
    "propn": "имена собственные",
    "psych": "психика и психические процессы",
    "put": "помещения и локализованные пространства",
    "qtm": "кванты и дискретные единицы",
    "quant": "количество и количественные характеристики",
    "set": "множества и совокупности объектов",
    "size": "размер и пространственные характеристики",
    "sound": "звук и акустические явления",
    "speech": "речь и речевая деятельность",
    "speed": "скорость движения или изменения",
    "stuff": "вещества и материалы",
    "taste": "вкус как сенсорная характеристика",
    "temper": "температура и тепловые характеристики",
    "text": "текстовые объекты и письменные данные",
    "time": "время и временные характеристики",
    "tool": "инструменты и орудия труда",
    "topon": "топонимы и географические названия",
    "transp": "транспорт и средства передвижения",
    "unit": "единицы измерения",
    "weapon": "оружие и военные средства",
    "weather": "погодные явления",
    "weight": "вес и масса объектов"
}


## 4. RuBERT, пакетная обработка и checkpoints

In [5]:
from __future__ import annotations

import hashlib
import os
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping, MutableMapping, Sequence

import numpy as np
from conllu import parse
from tqdm.auto import tqdm

DEFAULT_MODEL = "DeepPavlov/rubert-base-cased"
SKIPPED_UPOS = frozenset({"PUNCT", "SYM"})


def normalise_rows(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    norms = np.linalg.norm(values, axis=-1, keepdims=True)
    return values / np.maximum(norms, np.finfo(np.float32).eps)


def misc_tags(misc: Any, field: str = "sem") -> list[str]:
    if not misc or not isinstance(misc, Mapping):
        return []
    value = misc.get(field)
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [str(tag).strip() for tag in value if str(tag).strip()]
    return [tag.strip() for tag in str(value).strip("[]").split(",") if tag.strip()]


def signature(items: Sequence[str]) -> str:
    digest = hashlib.sha256()
    for item in items:
        digest.update(item.encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def atomic_pickle_dump(value: Any, path: Path) -> None:
    """Write a checkpoint atomically: an interrupted write never replaces a valid file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    temporary.replace(path)


@dataclass(frozen=True)
class ScoringConfig:
    context_tag_weight: float = 0.45
    definition_tag_weight: float = 0.45
    context_definition_weight: float = 0.10
    threshold: float = 0.50
    relative_margin: float = 0.08
    min_tags: int = 1
    max_tags: int = 5

    def __post_init__(self) -> None:
        weights = (self.context_tag_weight, self.definition_tag_weight,
                   self.context_definition_weight)
        if any(weight < 0 for weight in weights) or not any(weights):
            raise ValueError("Weights must be non-negative and not all zero")
        if self.min_tags < 0 or self.max_tags < self.min_tags:
            raise ValueError("Expected 0 <= min_tags <= max_tags")


class RuBERTEncoder:
    def __init__(self, model_name=DEFAULT_MODEL, *, cache_dir=None, device=None,
                 batch_size=32, max_length=256, use_amp=True,
                 local_files_only=False):
        import torch
        from transformers import AutoModel, AutoTokenizer

        self.torch = torch
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.batch_size = batch_size
        self.max_length = max_length
        self.use_amp = use_amp and self.device.type == "cuda"
        common = {"cache_dir": cache_dir, "local_files_only": local_files_only}
        print(f"Загрузка RuBERT на {self.device} (кэш: {cache_dir})")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, **common)
        self.model = AutoModel.from_pretrained(model_name, **common).to(self.device).eval()
        if not self.tokenizer.is_fast:
            raise ValueError("Для выравнивания слов нужен fast tokenizer")
        print("RuBERT загружен")

    def _forward(self, encoded: MutableMapping[str, Any]):
        torch = self.torch
        inputs = {key: value.to(self.device) for key, value in encoded.items()
                  if key != "offset_mapping"}
        with torch.inference_mode(), torch.autocast(
            device_type=self.device.type, enabled=self.use_amp
        ):
            return self.model(**inputs).last_hidden_state.float()

    def encode_texts(self, texts: Sequence[str]) -> np.ndarray:
        torch = self.torch
        if not texts:
            return np.empty((0, self.model.config.hidden_size), dtype=np.float32)
        chunks = []
        for start in range(0, len(texts), self.batch_size):
            batch = list(texts[start:start + self.batch_size])
            encoded = self.tokenizer(batch, padding=True, truncation=True,
                max_length=self.max_length, return_special_tokens_mask=True,
                return_tensors="pt")
            special = encoded.pop("special_tokens_mask").to(self.device)
            hidden = self._forward(encoded)
            mask = encoded["attention_mask"].to(self.device).bool() & ~special.bool()
            pooled = (hidden * mask.unsqueeze(-1)).sum(1)
            pooled /= mask.sum(1, keepdim=True).clamp_min(1)
            chunks.append(pooled.cpu().numpy())
        return normalise_rows(np.concatenate(chunks))

    def encode_contexts(self, windows: Sequence[Sequence[str]],
                        positions: Sequence[int]) -> np.ndarray:
        torch = self.torch
        if not windows:
            return np.empty((0, self.model.config.hidden_size), dtype=np.float32)
        texts, spans = [], []
        for tokens, target in zip(windows, positions):
            starts, cursor = [], 0
            for token in tokens:
                starts.append(cursor)
                cursor += len(str(token)) + 1
            texts.append(" ".join(map(str, tokens)))
            begin = starts[target]
            spans.append((begin, begin + len(str(tokens[target]))))
        chunks = []
        for start in range(0, len(texts), self.batch_size):
            batch_texts = texts[start:start + self.batch_size]
            batch_spans = spans[start:start + self.batch_size]
            encoded = self.tokenizer(batch_texts, padding=True, truncation=True,
                max_length=self.max_length, return_offsets_mapping=True,
                return_tensors="pt")
            offsets = encoded.pop("offset_mapping")
            hidden = self._forward(encoded)
            pooled = []
            for row, (begin, end) in enumerate(batch_spans):
                mask = (offsets[row, :, 0] < end) & (offsets[row, :, 1] > begin)
                if not mask.any():
                    raise ValueError(f"RuBERT обрезал целевое слово: {batch_texts[row]!r}")
                pooled.append(hidden[row, mask.to(self.device)].mean(0))
            chunks.append(torch.stack(pooled).cpu().numpy())
        return normalise_rows(np.concatenate(chunks))


def cached_encode_texts(texts: Sequence[str], encoder: RuBERTEncoder,
                        cache_dir: Path, name: str,
                        checkpoint_batch_size=1024) -> np.ndarray:
    """Encode texts in restartable chunks and show progress."""
    texts = list(texts)
    job_signature = signature(texts)
    job_dir = cache_dir / f"{name}_{job_signature[:12]}"
    job_dir.mkdir(parents=True, exist_ok=True)
    matrices = []
    progress = tqdm(total=len(texts), desc=f"RuBERT: {name}", unit="текст")
    for start in range(0, len(texts), checkpoint_batch_size):
        end = min(start + checkpoint_batch_size, len(texts))
        checkpoint = job_dir / f"{start:09d}_{end:09d}.npz"
        vector_batch = None
        if checkpoint.exists():
            try:
                with np.load(checkpoint) as saved:
                    if str(saved["signature"]) == job_signature:
                        vector_batch = saved["vectors"].astype(np.float32)
            except Exception:
                vector_batch = None
        if vector_batch is None:
            vector_batch = encoder.encode_texts(texts[start:end])
            temporary = checkpoint.with_suffix(".tmp.npz")
            np.savez_compressed(temporary, signature=job_signature,
                                vectors=vector_batch.astype(np.float16))
            temporary.replace(checkpoint)
        matrices.append(vector_batch)
        progress.update(end - start)
        progress.set_postfix(checkpoint=checkpoint.name)
    progress.close()
    if not matrices:
        return np.empty((0, encoder.model.config.hidden_size), dtype=np.float32)
    return normalise_rows(np.concatenate(matrices))


class SemanticAnnotator:
    def __init__(self, encoder, tag_descriptions, dictionary, *, cache_dir,
                 config=None, context_window=5, checkpoint_batch_size=1024):
        self.encoder = encoder
        self.config = config or ScoringConfig()
        self.context_window = context_window
        self.tags = tuple(tag_descriptions)
        tag_texts = [f"Семантический тег {tag}: {tag_descriptions[tag]}"
                     for tag in self.tags]
        self.tag_vectors = cached_encode_texts(
            tag_texts, encoder, cache_dir, "tags", checkpoint_batch_size
        )
        self.definitions = {
            str(lemma).lower(): tuple(dict.fromkeys(
                text for text in map(str, definitions) if text.strip()))
            for lemma, definitions in dictionary.items() if definitions
        }
        unique = list(dict.fromkeys(text for values in self.definitions.values()
                                    for text in values))
        vectors = cached_encode_texts(
            unique, encoder, cache_dir, "definitions", checkpoint_batch_size
        )
        self.definition_vectors = dict(zip(unique, vectors))

    @staticmethod
    def dictionary_from_rows(rows: Iterable[Mapping[str, Any]], vocabulary=None):
        vocabulary = {word.lower() for word in vocabulary} if vocabulary else None
        result = {}
        progress = tqdm(rows, total=len(rows), desc="Отбор словарных статей", unit="статья")
        for row in progress:
            word = str(row.get("word", "")).strip().lower()
            definition = str(row.get("definition", "")).strip()
            if word and definition and (vocabulary is None or word in vocabulary):
                result.setdefault(word, []).append(definition)
        progress.close()
        print(f"Найдено словарных лемм: {len(result):,}")
        return result

    @staticmethod
    def candidates(sentence):
        return [i for i, token in enumerate(sentence)
                if isinstance(token.get("id"), int)
                and token.get("upos") not in SKIPPED_UPOS
                and any(char.isalnum() for char in str(token.get("form", "")))]

    def contexts(self, sentence, candidates):
        windows, positions = [], []
        for index in candidates:
            left = max(0, index - self.context_window)
            right = min(len(sentence), index + self.context_window + 1)
            windows.append([str(token.get("form", "")) for token in sentence[left:right]])
            positions.append(index - left)
        return windows, positions

    def select_tags(self, context_vector, definition_vector):
        cfg = self.config
        context_tag = self.tag_vectors @ context_vector
        if definition_vector is None:
            scores = context_tag
        else:
            scores = (cfg.context_tag_weight * context_tag
                + cfg.definition_tag_weight * (self.tag_vectors @ definition_vector)
                + cfg.context_definition_weight * float(context_vector @ definition_vector))
            scores /= (cfg.context_tag_weight + cfg.definition_tag_weight
                       + cfg.context_definition_weight)
        order = np.argsort(-scores)
        best = float(scores[order[0]])
        selected = [int(i) for i in order
                    if scores[i] >= cfg.threshold
                    and scores[i] >= best - cfg.relative_margin][:cfg.max_tags]
        if len(selected) < cfg.min_tags:
            selected = [int(i) for i in order[:cfg.min_tags]]
        return ([self.tags[i] for i in selected],
                {tag: float(score) for tag, score in zip(self.tags, scores)})

    def apply_vectors(self, sentence, candidates, context_vectors):
        rows = []
        for index, context_vector in zip(candidates, context_vectors):
            token = sentence[index]
            gold = misc_tags(token.get("misc"))
            lemma = str(token.get("lemma") or token.get("form") or "").lower()
            definitions = self.definitions.get(lemma, ())
            best_definition = None
            definition_similarity = None
            definition_vector = None
            if definitions:
                matrix = np.stack([self.definition_vectors[text] for text in definitions])
                similarities = matrix @ context_vector
                best_index = int(np.argmax(similarities))
                best_definition = definitions[best_index]
                definition_similarity = float(similarities[best_index])
                definition_vector = matrix[best_index]
            tags, scores = self.select_tags(context_vector, definition_vector)
            misc = dict(token.get("misc") or {})
            misc["sem"] = f"[{','.join(tags)}]"
            token["misc"] = misc
            rows.append({"id": token.get("id"), "form": token.get("form"),
                "lemma": lemma, "gold_tags": gold, "predicted_tags": tags,
                "definition": best_definition,
                "definition_similarity": definition_similarity,
                "tag_scores": scores,
                "status": "classified" if definitions else "classified_without_definition"})
        return rows

    def annotate_with_checkpoints(self, conllu_text, checkpoint_dir,
                                  sentences_per_checkpoint=100):
        sentences = parse(conllu_text)
        source_signature = signature([conllu_text])
        run_dir = Path(checkpoint_dir) / f"corpus_{source_signature[:12]}"
        run_dir.mkdir(parents=True, exist_ok=True)
        fragments, diagnostics = [], []
        progress = tqdm(total=len(sentences), desc="Разметка корпуса", unit="предл.")
        for start in range(0, len(sentences), sentences_per_checkpoint):
            end = min(start + sentences_per_checkpoint, len(sentences))
            checkpoint = run_dir / f"sentences_{start:07d}_{end:07d}.pkl"
            saved = None
            if checkpoint.exists():
                try:
                    with checkpoint.open("rb") as file:
                        saved = pickle.load(file)
                    if saved.get("signature") != source_signature:
                        saved = None
                except Exception:
                    saved = None
            if saved is None:
                batch_sentences = sentences[start:end]
                all_windows, all_positions, candidate_lists = [], [], []
                for sentence in batch_sentences:
                    candidates = self.candidates(sentence)
                    candidate_lists.append(candidates)
                    windows, positions = self.contexts(sentence, candidates)
                    all_windows.extend(windows)
                    all_positions.extend(positions)
                vectors = self.encoder.encode_contexts(all_windows, all_positions)
                offset, batch_rows = 0, []
                for local_index, (sentence, candidates) in enumerate(
                    zip(batch_sentences, candidate_lists)
                ):
                    count = len(candidates)
                    rows = self.apply_vectors(
                        sentence, candidates, vectors[offset:offset + count]
                    )
                    offset += count
                    batch_rows.extend({"sentence_index": start + local_index, **row}
                                      for row in rows)
                saved = {"signature": source_signature,
                         "conllu": "".join(s.serialize() for s in batch_sentences),
                         "diagnostics": batch_rows}
                atomic_pickle_dump(saved, checkpoint)
            fragments.append(saved["conllu"])
            diagnostics.extend(saved["diagnostics"])
            progress.update(end - start)
            progress.set_postfix(checkpoint=checkpoint.name,
                                 words=len(diagnostics))
        progress.close()
        return "".join(fragments), diagnostics


## 5. Подготовка рабочего словаря

Из большого словаря выбираются только леммы входного корпуса. Это существенно сокращает время BERT-разметки и размер checkpoints.


In [6]:
sentences_for_vocabulary = parse(conllu_data)
corpus_lemmas = {
    str(token.get('lemma') or token.get('form') or '').lower()
    for sentence in sentences_for_vocabulary
    for token in sentence
    if isinstance(token.get('id'), int)
}
print(f'Уникальных лемм в корпусе: {len(corpus_lemmas):,}')

dictionary = SemanticAnnotator.dictionary_from_rows(
    dictionary_definitions['train'], vocabulary=corpus_lemmas
)


Уникальных лемм в корпусе: 475


Отбор словарных статей:   0%|          | 0/294007 [00:00<?, ?статья/s]

Найдено словарных лемм: 441


## 6. Инициализация модели и векторизация

При первом запуске индикаторы `RuBERT: tags` и `RuBERT: definitions` показывают обработку. При повторном запуске готовые пакеты читаются из Drive.


In [7]:
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
BATCH_SIZE = 32                 # уменьшите до 8–16 при нехватке GPU-памяти
CHECKPOINT_TEXTS = 512          # сохранять векторы каждые 512 текстов
SENTENCES_PER_CHECKPOINT = 50   # сохранять разметку каждые 50 предложений

encoder = RuBERTEncoder(
    MODEL_NAME,
    cache_dir=str(MODEL_CACHE),
    batch_size=BATCH_SIZE,
    max_length=256,
)
print(f'Устройство: {encoder.device}')

annotator = SemanticAnnotator(
    encoder, TAGS, dictionary,
    cache_dir=VECTOR_CACHE,
    checkpoint_batch_size=CHECKPOINT_TEXTS,
    context_window=5,
    config=ScoringConfig(
        context_tag_weight=0.45,
        definition_tag_weight=0.45,
        context_definition_weight=0.10,
        threshold=0.50,
        relative_margin=0.08,
        min_tags=1,
        max_tags=5,
    ),
)


Загрузка RuBERT на cpu (кэш: /content/drive/MyDrive/Colab Notebooks/Аспирантура/rubert_semantic_checkpoints/huggingface_models)


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuBERT загружен
Устройство: cpu


RuBERT: tags:   0%|          | 0/82 [00:00<?, ?текст/s]

RuBERT: definitions:   0%|          | 0/3123 [00:00<?, ?текст/s]

## 7. Разметка с автоматическим продолжением

Если соединение оборвалось, снова выполните ячейки сверху. Уже готовые пакеты предложений будут пропущены. Выходной файл обновляется после успешной сборки всех пакетов.


In [8]:
annotated_conllu, diagnostics = annotator.annotate_with_checkpoints(
    conllu_data,
    checkpoint_dir=CORPUS_CACHE,
    sentences_per_checkpoint=SENTENCES_PER_CHECKPOINT,
)

OUTPUT_FILE.write_text(annotated_conllu, encoding='utf-8')
atomic_pickle_dump(diagnostics, DIAGNOSTICS_FILE)
results = pd.DataFrame(diagnostics)
print(f'Готово. Размечено слов: {len(results):,}')
print('CoNLL-U:', OUTPUT_FILE)
print('Диагностика:', DIAGNOSTICS_FILE)
results[['form', 'lemma', 'definition', 'definition_similarity',
         'predicted_tags', 'status']].head(20)


Разметка корпуса:   0%|          | 0/82 [00:00<?, ?предл./s]

Готово. Размечено слов: 865
CoNLL-U: /content/drive/MyDrive/Colab Notebooks/Аспирантура/conllu/handlabeled/GramEval2020-test-fiction-rubert-sem.conllu
Диагностика: /content/drive/MyDrive/Colab Notebooks/Аспирантура/conllu/handlabeled/GramEval2020-test-fiction-rubert-sem.pkl


,form,lemma,definition,definition_similarity,predicted_tags,status
0,С,с,с род. и твор. пад. употр. при выражении объек...,0.291286,[degr],classified
1,некоторых,некоторый,"кое-какой, не очень значительный",0.374093,[posit],classified
2,пор,пора,о наступлении срока чего-л,0.175482,[appear],classified
3,я,я,"употр. при обозначении одного, отдельного чело...",0.214926,[poss],classified
4,полюбил,полюбить,"почувствовать горячую сердечную склонность, вл...",0.294202,[taste],classified
5,это,этот,"тот, кто находится в непосредственной близости...",0.329081,[appear],classified
6,небольшое,небольшой,занимающий невысокое положение в обществе,0.424195,[min],classified
7,кафе,кафе,небольшой ресторан,0.543222,[cin],classified
8,Может,мочь,иметь возможность делать что-л,0.214993,[appear],classified
9,быть,быть,разг. прибывать (о транспорте); 4) происходить...,0.246293,[degr],classified


## 8. Метрики на токенах с эталонной разметкой

In [9]:
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer

evaluated = results[results['gold_tags'].map(bool)].copy()
if evaluated.empty:
    print('В исходном корпусе нет gold-тегов для оценки.')
else:
    labels = sorted(TAGS)
    mlb = MultiLabelBinarizer(classes=labels)
    y_true = mlb.fit_transform(evaluated['gold_tags'])
    y_pred = mlb.transform(evaluated['predicted_tags'])
    print(classification_report(y_true, y_pred, target_names=labels,
                                zero_division=0))


              precision    recall  f1-score   support

       abstr       0.00      0.00      0.00        79
         age       0.00      0.00      0.00         0
      animal       0.17      0.33      0.22         3
      appear       0.03      0.60      0.06         5
         art       0.00      0.00      0.00         0
     artwork       0.00      0.00      0.00         0
          be       0.40      0.12      0.18        17
       behav       0.00      0.00      0.00         1
       berry       0.00      0.00      0.00         0
    changest       0.00      0.00      0.00         6
         cin       0.00      0.00      0.00         0
       class       0.00      0.00      0.00         5
       cloth       0.05      1.00      0.10         1
       color       0.00      0.00      0.00         1
       concr       0.00      0.00      0.00       186
      constr       0.00      0.00      0.00         2
     contact       0.00      0.00      0.00         0
        degr       0.08    

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['sport'] will be ignored
  warnings.warn(
